In [7]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
pio.templates.default = 'plotly_white'

In [3]:
data = pd.read_csv('supply_chain_data.csv')
data

,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,Mumbai,29,215,29,46.279879,Pending,0.226410,Road,Route B,187.752075
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,Mumbai,23,517,30,33.616769,Pending,4.854068,Road,Route B,503.065579
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,Mumbai,12,971,27,30.688019,Pending,4.580593,Air,Route C,141.920282
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,Kolkata,24,937,18,35.624741,Fail,4.746649,Rail,Route A,254.776159
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,Delhi,5,414,3,92.065161,Fail,3.145580,Air,Route A,923.440632
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,haircare,SKU95,77.903927,65,672,7386.363944,Unknown,15,14,26,...,Mumbai,18,450,26,58.890686,Pending,1.210882,Air,Route A,778.864241
96,cosmetics,SKU96,24.423131,29,324,7698.424766,Non-binary,67,2,32,...,Mumbai,28,648,28,17.803756,Pending,3.872048,Road,Route A,188.742141
97,haircare,SKU97,3.526111,56,62,4370.916580,Male,46,19,4,...,Mumbai,10,535,13,65.765156,Fail,3.376238,Road,Route A,540.132423
98,skincare,SKU98,19.754605,43,913,8525.952560,Female,53,1,27,...,Chennai,28,581,9,5.604691,Pending,2.908122,Rail,Route A,882.198864


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Product type             100 non-null    object 
 1   SKU                      100 non-null    object 
 2   Price                    100 non-null    float64
 3   Availability             100 non-null    int64  
 4   Number of products sold  100 non-null    int64  
 5   Revenue generated        100 non-null    float64
 6   Customer demographics    100 non-null    object 
 7   Stock levels             100 non-null    int64  
 8   Lead times               100 non-null    int64  
 9   Order quantities         100 non-null    int64  
 10  Shipping times           100 non-null    int64  
 11  Shipping carriers        100 non-null    object 
 12  Shipping costs           100 non-null    float64
 13  Supplier name            100 non-null    object 
 14  Location                 10

In [14]:
#let’s get started with analyzing the Supply Chain by looking at the relationship between the price of the products and the revenue generated by them
fig = px.scatter(data, x='Price',y='Revenue generated', color = 'Product type',
                hover_data=['Number of products sold'], trendline='ols')
fig.show()

In [21]:
#let’s have a look at the sales by product type
sales = data.groupby('Product type')['Number of products sold'].sum().reset_index()
fig1 = px.pie(sales, names='Product type', values='Number of products sold', hole=0.5,title = 'Sales by product type',color_discrete_sequence=px.colors.qualitative.Pastel)
fig1.update_traces(title='Products sold', textposition='inside',textinfo='label+percent')
fig1.show()

In [26]:
# Now let’s have a look at the total revenue generated from shipping carriers
sc = data.groupby('Shipping carriers')['Revenue generated'].sum().reset_index()

fig2 = px.pie(sc, values = 'Revenue generated', names='Shipping carriers', hole=0.5, title='Revenue Generated By Shipping Carriers')
fig2.update_traces(title='Revenue generated', textposition='inside', textinfo='label+percent')
fig2.show()

In [28]:
# Now let’s have a look at the Average lead time and Average Manufacturing Costs for all products of the company
avg_lt =data.groupby('Product type')['Lead time'].mean().reset_index()
avg_mc = data.groupby('Product type')['Manufacturing costs'].mean().reset_index()

result = pd.merge(avg_lt, avg_mc, on='Product type')
result

,Product type,Lead time,Manufacturing costs
0,cosmetics,13.538462,43.052740
1,haircare,18.705882,48.457993
2,skincare,18.000000,48.993157


In [31]:
sku_rev = data.groupby('SKU')['Revenue generated'].sum().reset_index()
# fig3 = px.line(sku_rev, x='SKU', y='Revenue generated')
fig3 = px.bar(sku_rev,x='SKU',y='Revenue generated')
fig3.show()

In [32]:
sku_stock = data.groupby('SKU')['Stock levels'].sum().reset_index()
fig4 = px.bar(sku_stock, x='SKU', y='Stock levels',title='Stock Available for each SKU')
fig4.show()

In [33]:
# Now let’s have a look at the order quantity of each SKU
sku_orders = data.groupby('SKU')['Order quantities'].sum().reset_index()

fig5 = px.bar(sku_orders, x='SKU', y='Order quantities', color='SKU',title='Order Quantities depending on the SKU')
fig5.show()

In [36]:
# Now let’s analyze the shipping cost of Carriers
fig6 = px.bar(data, x='Shipping carriers',y='Shipping costs', title='Shipping Cost depending on the carriers')
fig6.show()

In [38]:
# Now let’s have a look at the cost distribution by transportation mode
fig7 = px.pie(data, values='Costs', names='Transportation modes', hole=0.5, title='Distribution by transportation mode')
fig7.show()

In [40]:
# Now let’s have a look at the defect rates by mode of transportation
# dt = data.groupby('Transportation modes')['Defect rates'].sum().reset_index()
# fig8 = px.pie(dt, values='Defect rates', names='Transportation modes', hole=0.5)
# fig8.show()
pivot_table = pd.pivot_table(data, values='Defect rates', 
                             index=['Transportation modes'], 
                             aggfunc='mean')

transportation_chart = px.pie(values=pivot_table["Defect rates"], 
                              names=pivot_table.index, 
                              title='Defect Rates by Transportation Mode',
                              hole=0.5,
                              color_discrete_sequence=px.colors.qualitative.Pastel)
transportation_chart.show()